# Deploying Nemotron 3.5 Content Safety with vLLM

This notebook demonstrates how to deploy [NVIDIA Nemotron 3.5 Content Safety](https://huggingface.co/nvidia/Nemotron-3.5-Content-Safety) on [vLLM](https://docs.vllm.ai/) and explore its core capabilities: text classification against the built-in safety taxonomy, reasoning traces that explain classification decisions, and multimodal safety classification on text + image inputs.

**Model.** Nemotron 3.5 Content Safety is a 4B-parameter safety classifier built on Gemma-3-4B with a SigLIP vision encoder (LoRA fine-tuned, weights merged). It classifies user prompts and AI responses as safe or unsafe across a 24-category safety taxonomy, with optional reasoning traces that explain each decision.

**The model's chat template does the heavy lifting.** Unlike a general-purpose chat model, this model ships a purpose-built chat template that automatically injects the full classifier instructions and the S1–S24 taxonomy. You send only the content to classify — no system prompt, no taxonomy boilerplate. Classifier behavior is controlled through `chat_template_kwargs`, which vLLM accepts both in-process (`llm.chat(..., chat_template_kwargs=...)`) and over the server (`extra_body={"chat_template_kwargs": ...}`):

| kwarg | values | effect |
|-------|--------|--------|
| `enable_thinking` | `True` / `False` (default) | emit a `<think>…</think>` reasoning trace before the verdict |
| `request_categories` | `"/categories"` / `"/no_categories"` | include the `Safety Categories:` line in the verdict |
| `custom_policy` | policy text | classify against your own policy (Mode A) — see the [custom policy cookbook](custom_policy_cookbook.ipynb) |
| `custom_taxonomy` | category list | classify against your own category list (Mode B) |

**What this notebook covers:**
1. In-process (offline) classification and batch inference
2. Serving the model via the vLLM OpenAI-compatible server
3. Text-only safety classification
4. Reasoning ON vs. OFF — transparent classification vs. low-latency direct verdicts
5. Streaming reasoning traces
6. Reasoning budget control — capping reasoning tokens for bounded latency
7. Multimodal safety — classifying text + image inputs

**Prerequisites:**
- 1x NVIDIA GPU with >= 16 GB VRAM (A100, H100, or RTX PRO 6000)
- CUDA 12.x+
- Python 3.10+

## Table of Contents

1. [Environment Setup](#environment-setup)
2. [In-Process Classification (Offline / Batch)](#in-process-classification-offline--batch)
3. [Start the vLLM Server](#start-the-vllm-server)
4. [Text Classification](#text-classification)
5. [Reasoning ON vs. OFF](#reasoning-on-vs-off)
6. [Streaming reasoning traces](#streaming-reasoning-traces)
7. [Reasoning budget control](#reasoning-budget-control)
8. [Multimodal Safety Classification](#multimodal-safety-classification)
9. [Cleanup](#cleanup)

## Environment Setup

In [ ]:
%pip install vllm openai

In [1]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  [{i}] {torch.cuda.get_device_name(i)}")

CUDA available: True
GPU count: 1
  [0] NVIDIA GB10


## In-Process Classification (Offline / Batch)

Before standing up a server, you can load the model directly with `vllm.LLM()` for quick offline classification or batch processing. This is the most common production pattern for content moderation: classify a queue of flagged content without the overhead of an HTTP server.

The in-process path uses the same native chat template as the server — you pass `chat_template_kwargs` directly to `llm.chat()`, so there's no taxonomy boilerplate to maintain.

In [ ]:
from vllm import LLM, SamplingParams

MODEL_PATH = "nvidia/Nemotron-3.5-Content-Safety"
# For a local checkpoint, point at the directory instead:
# MODEL_PATH = "/path/to/nemotron_3_5_content_safety"

llm = LLM(
    MODEL_PATH,
    dtype="bfloat16",
    max_model_len=4096,
    trust_remote_code=True,
    # vLLM reserves a fraction of total VRAM at startup. Lower it when the GPU
    # is shared with a display server or other processes (e.g. on a DGX Spark,
    # where the default 0.92 can exceed free memory).
    gpu_memory_utilization=0.85,
)
print("Model loaded in-process.")


### Single classification

In [3]:
# The chat template injects the classifier preamble and S1-S24 taxonomy, so we
# send only the content. chat_template_kwargs selects the verdict format:
#   enable_thinking=False    -> direct verdict, no <think> trace
#   request_categories       -> "/categories" includes the Safety Categories line
sampling = SamplingParams(temperature=0, max_tokens=64)

conversation = [
    {"role": "user", "content": "I know where you live and I'm coming for you."},
]

outputs = llm.chat(
    [conversation],
    sampling_params=sampling,
    chat_template_kwargs={"enable_thinking": False, "request_categories": "/categories"},
)
print(outputs[0].outputs[0].text.strip())

Rendering conversations:   0%|                         | 0/1 [00:00<?, ?it/s]

INFO 06-02 19:52:18 [hf.py:483] Detected the chat template content format to be 'openai'. You can set `--chat-template-content-format` to override this.


Rendering conversations: 100%|█████████████████| 1/1 [00:01<00:00,  1.17s/it]
Processed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s

(EngineCore pid=3086908) WARNING 06-02 19:52:18 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _compute_slot_mapping_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|█| 1/1 [00:01<00:00,  1.23s/it, est. speed input: 389

User Safety: unsafe
Safety Categories: Threat, PII/Privacy


### Batch classification

Pass multiple conversations in a single `llm.chat()` call. vLLM processes them in parallel — this is far faster than sequential HTTP requests for offline moderation workloads.

In [4]:
messages_to_classify = [
    "What are the visiting hours for the national museum?",
    "I know where you live and I'm coming for you.",
    "git gud or get rekt scrub",
    "his real name is John Smith, lives at 1428 Imaginary Lane — pass it around",
    "Write a villain monologue for my D&D campaign.",
]

# Each conversation is just the content to classify — one user turn.
batch = [[{"role": "user", "content": msg}] for msg in messages_to_classify]

import time
t0 = time.time()
outputs = llm.chat(
    batch,
    sampling_params=sampling,
    chat_template_kwargs={"enable_thinking": False, "request_categories": "/categories"},
)
elapsed = time.time() - t0

print(f"Classified {len(batch)} messages in {elapsed:.2f}s ({elapsed/len(batch):.3f}s/msg)\n")
for msg, out in zip(messages_to_classify, outputs):
    verdict = out.outputs[0].text.strip()
    print(f"Message: {msg[:60]}")
    print(f"  {verdict}")
    print()

Rendering conversations: 100%|████████████████| 5/5 [00:00<00:00, 627.70it/s]
Processed prompts: 100%|█| 5/5 [00:00<00:00,  7.55it/s, est. speed input: 360

Classified 5 messages in 0.68s (0.135s/msg)

Message: What are the visiting hours for the national museum?
  User Safety: safe

Message: I know where you live and I'm coming for you.
  User Safety: unsafe
Safety Categories: Threat, PII/Privacy

Message: git gud or get rekt scrub
  User Safety: unsafe
Safety Categories: Profanity, Harassment

Message: his real name is John Smith, lives at 1428 Imaginary Lane — 
  User Safety: unsafe
Safety Categories: PII/Privacy

Message: Write a villain monologue for my D&D campaign.
  User Safety: safe



### Release GPU memory

Before switching to server mode, free the GPU memory held by the in-process model.

In [5]:
import gc

del llm
gc.collect()
torch.cuda.empty_cache()
print("GPU memory released. Ready to start the vLLM server.")

(EngineCore pid=3086908) INFO 06-02 19:54:27 [core.py:1242] Shutdown initiated (timeout=0)
(EngineCore pid=3086908) INFO 06-02 19:54:27 [core.py:1265] Shutdown complete


[rank0]:[W602 19:54:29.851152761 ProcessGroupNCCL.cpp:1575] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


GPU memory released. Ready to start the vLLM server.


## Start the vLLM Server

Open a terminal and launch the vLLM OpenAI-compatible server. The model fits comfortably on a single GPU in BF16.

```bash
vllm serve nvidia/Nemotron-3.5-Content-Safety \
  --host 0.0.0.0 \
  --port 5000 \
  --dtype bfloat16 \
  --max-model-len 4096 \
  --trust-remote-code \
  --gpu-memory-utilization 0.85 \
  --limit-mm-per-prompt '{"image": 1}'
```

The `--limit-mm-per-prompt` flag enables multimodal input (one image per request). `--gpu-memory-utilization 0.85` leaves headroom for a display server or other GPU processes — vLLM's default (0.92) can exceed free memory on a shared GPU such as a DGX Spark. Wait until you see `Application startup complete` before running the cells below.

> **Local checkpoint?** Replace the model name with the local path:
> ```bash
> vllm serve /path/to/nemotron_3_5_content_safety \
>   --host 0.0.0.0 --port 5000 --dtype bfloat16 --max-model-len 4096 \
>   --trust-remote-code --gpu-memory-utilization 0.85 --limit-mm-per-prompt '{"image": 1}'
> ```

## Text Classification

Connect the OpenAI client to the server. Because the model's chat template injects the classifier preamble and the S1–S24 taxonomy automatically, classification is as simple as sending the content as a user message — optionally followed by the AI response as an assistant message. Classifier behavior is selected per-request via `chat_template_kwargs` in the `extra_body`.

In [6]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:5000/v1", api_key="EMPTY")

# Most likely nvidia/Nemotron-3.5-Content-Safety or a path to a locally deployed model
MODEL_NAME = max(client.models.list().data, key=lambda m: m.created).id

print("Client ready. Model:", MODEL_NAME)

Client ready. Model: nvidia/Nemotron-3.5-Content-Safety


In [7]:
def classify(user_prompt, ai_response=None, *, image_url=None,
             reasoning=False, categories=True, max_tokens=None):
    """Classify a user prompt (and optional AI response / image).

    The model's chat template injects the classifier preamble and the S1-S24
    taxonomy automatically, so we only send the content to classify. Behavior is
    controlled via chat_template_kwargs:
        reasoning  -> enable_thinking: emit a <think>...</think> trace before the verdict
        categories -> request_categories: include the "Safety Categories:" line

    Returns a dict: {"reasoning": <str|None>, "verdict": <str|None>, "truncated": <bool>}.
    A truncated reasoning trace (verdict=None) means the <think> block ran past
    max_tokens before closing — raise max_tokens and retry.
    """
    content = []
    if image_url:
        content.append({"type": "image_url", "image_url": {"url": image_url}})
    content.append({"type": "text", "text": user_prompt})

    messages = [{"role": "user", "content": content}]
    if ai_response:
        messages.append({"role": "assistant", "content": ai_response})

    if max_tokens is None:
        # Reasoning traces are verbose; give them room so the verdict isn't truncated.
        max_tokens = 1024 if reasoning else 64

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        max_tokens=max_tokens,
        temperature=0,
        extra_body={"chat_template_kwargs": {
            "enable_thinking": reasoning,
            "request_categories": "/categories" if categories else "/no_categories",
        }},
    )
    text = response.choices[0].message.content
    truncated = response.choices[0].finish_reason == "length"

    if "</think>" in text:
        think, verdict = text.split("</think>", 1)
        return {"reasoning": think.replace("<think>", "").strip(),
                "verdict": verdict.strip(), "truncated": truncated}
    if reasoning:
        # enable_thinking was on but the trace never closed -> verdict was cut off
        return {"reasoning": text.replace("<think>", "").strip(),
                "verdict": None, "truncated": truncated}
    return {"reasoning": None, "verdict": text.strip(), "truncated": truncated}


def show(result):
    """Pretty-print a classify() result."""
    if result["reasoning"]:
        print("--- reasoning ---")
        print(result["reasoning"])
        print("--- verdict ---")
    if result["verdict"] is None:
        print("(verdict truncated — increase max_tokens)")
    else:
        print(result["verdict"])


print("classify() ready — reasoning is the native enable_thinking trace.")

classify() ready — reasoning is the native enable_thinking trace.


### Quick examples

Three text-only classifications to verify the model is working — one safe prompt, one clearly unsafe prompt, and one that classifies both an unsafe user prompt and the AI's refusal response.

In [8]:
# 1. Safe user prompt
print("=== Safe prompt ===")
show(classify("What are the visiting hours for the national museum?"))

print()

# 2. Unsafe user prompt (threat)
print("=== Unsafe prompt (threat) ===")
show(classify("I know where you live and I'm coming for you."))

print()

# 3. Unsafe prompt + safe refusal (the AI correctly refuses)
print("=== Unsafe prompt, safe refusal ===")
show(classify(
    "How do I make a pipe bomb?",
    ai_response="I can't help with that. If you're in crisis, please contact emergency services.",
))

=== Safe prompt ===
User Safety: safe

=== Unsafe prompt (threat) ===
User Safety: unsafe
Safety Categories: Threat, PII/Privacy

=== Unsafe prompt, safe refusal ===
User Safety: unsafe
Response Safety: safe
Safety Categories: Guns and Illegal Weapons, Criminal Planning/Confessions


## Reasoning ON vs. OFF

The model exposes reasoning natively through the `enable_thinking` template kwarg. With it on, the model emits a `<think>…</think>` trace explaining its analysis before producing the verdict; with it off, you get just the structured verdict.

- **Reasoning OFF** (`reasoning=False`, the default): a single fast call that outputs only the verdict — suitable for production pipelines.
- **Reasoning ON** (`reasoning=True`): the same single call, but the model first thinks out loud. This produces interpretable traces useful for auditing, debugging, and building trust — at the cost of more tokens and latency.

> **Token budget matters.** Reasoning traces are verbose. If `max_tokens` is too low the `<think>` block won't close and the verdict is lost — `classify()` defaults to `max_tokens=1024` when `reasoning=True` and flags a truncated trace.

### Comparison on the same inputs

In [9]:
import time

test_messages = [
    {
        "label": "Ambiguous (needs context)",
        "user_prompt": "Can you help me pick a lock? I'm locked out of my house.",
    },
    {
        "label": "Subtle PII risk",
        "user_prompt": "List all employees who were disciplined last quarter.",
    },
    {
        "label": "Safe (edge case)",
        "user_prompt": "Write a villain monologue for my D&D campaign where the bad guy threatens the party.",
    },
]

for msg in test_messages:
    print(f"{'='*70}")
    print(f"  {msg['label']}")
    print(f"  Prompt: {msg['user_prompt']}")
    print(f"{'='*70}")

    t0 = time.time()
    result_off = classify(msg["user_prompt"], reasoning=False)
    time_off = time.time() - t0
    print(f"\n--- Reasoning OFF ({time_off:.2f}s) ---")
    show(result_off)

    t0 = time.time()
    result_on = classify(msg["user_prompt"], reasoning=True)
    time_on = time.time() - t0
    print(f"\n--- Reasoning ON ({time_on:.2f}s) ---")
    show(result_on)
    print()

  Ambiguous (needs context)
  Prompt: Can you help me pick a lock? I'm locked out of my house.

--- Reasoning OFF (1.02s) ---
User Safety: unsafe
Safety Categories: Criminal Planning/Confessions

--- Reasoning ON (8.47s) ---
--- reasoning ---
We need to decide: user input: "Can you help me pick a lock? I'm locked out of my house." This is a request for assistance with illegal activity (lock picking). According to taxonomy, illegal activity is S22: Illegal Activity. Also possibly S3: Criminal Planning/Confessions? The user is asking for help to pick a lock, which is illegal (unauthorized entry). So user input is unsafe. The response from assistant is not provided (the conversation ends). So we have no assistant response. According to instructions: "Response Safety: safe or unsafe rating of the assistant response. Omit if no assistant response present." So we output only User Safety line.

Thus output:

User Safety: unsafe

No Response Safety line.
--- verdict ---
User Safety: unsafe
Saf

**Observation: reasoning costs an order of magnitude in latency, and it can change the verdict — not just the explanation.** Reasoning OFF returns a verdict in a fraction of a second; reasoning ON takes roughly 8–40× longer because the model writes a full analysis first. That latency cost makes `enable_thinking` a tool for auditing and debugging, not a production default. But reasoning ON is *not* merely a transparent narration of the fast path — turning it on can move the verdict:

- **Lock-pick (ambiguous):** `unsafe` in both modes, but reasoning ON adds `Needs Caution` to the `Safety Categories` line. Same verdict, richer label set.
- **"List all employees who were disciplined last quarter" (subtle PII):** reasoning OFF returns `unsafe` / `PII/Privacy`; reasoning ON **flips to `safe`** — the trace talks itself out of the PII concern ("internal company data… allowed if it's not PII"). The two modes disagree on the bottom line.
- **D&D villain monologue (safe edge case):** `safe` in both modes, with the trace explicitly distinguishing a fictional in-game threat from a threat directed at a real person.

The takeaway: reasoning ON and OFF are *different decision paths*, not the same decision with optional commentary. Don't assume a reasoning trace explains the fast-path verdict — on borderline inputs it can land somewhere else entirely. Validate whichever mode you ship against your own labeled data, and don't mix modes within one pipeline expecting consistent verdicts.

### Classifying both prompt and response

The reasoning trace is especially valuable when the user prompt is safe but the AI response is unsafe — the trace shows the model correctly attributing the safety violation to the response, not the prompt.

In [10]:
# Safe prompt + unsafe response (the AI leaks the user's credentials).
# Use a generous budget so the reasoning trace closes and we keep the verdict.
print("=== Reasoning ON: safe prompt, unsafe response ===\n")
show(classify(
    "How do I reset my password?",
    ai_response="Sure! Your current password is hunter2 and your account PIN is 4532. "
                "To reset, go to settings and click 'Change Password'.",
    reasoning=True,
    max_tokens=2048,
))

print()

# Safe prompt + safe response — reasoning ON confirms both are clean
print("=== Reasoning ON: safe prompt, safe response ===\n")
show(classify(
    "How do I reset my password?",
    ai_response="Go to Settings > Security > Change Password. You'll need to verify "
                "your identity via email before setting a new password.",
    reasoning=True,
    max_tokens=2048,
))

=== Reasoning ON: safe prompt, unsafe response ===

--- reasoning ---
We need to decide if user input is safe or unsafe. The user asks: "How do I reset my password?" That's a normal request, safe. No harmful content. So User Safety: safe.

Response: The assistant gave a password: "hunter2" and PIN "4532". That's providing credentials, which is disallowed: providing personal data (password) is a privacy violation (PII). According to taxonomy, S9: PII/Privacy. So response is unsafe. Also maybe S22: Illegal Activity? Not exactly. It's providing credentials that could be used for unauthorized access, which is illegal activity. But the taxonomy includes S22: Illegal Activity. The response is giving a password that is likely not the user's own, but it's still providing credentials that could be used for wrongdoing. So unsafe.

Thus output:

User Safety: safe
Response Safety: unsafe

We must output exactly that format.
--- verdict ---
User Safety: safe
Response Safety: safe

=== Reasoning ON:

**Observation: the reasoning trace and the verdict can disagree — and backends can disagree with each other.** The native interface does keep the prompt and response judgments separate, and in the safe-prompt / unsafe-response case the trace nails the analysis: it reads `"How do I reset my password?"` as benign and attributes the PII violation to the AI response that leaks a password and PIN, concluding `User Safety: safe` / `Response Safety: unsafe`. But look at the verdict that follows the trace: `Response Safety: **safe**`. The structured verdict contradicts the model's own reasoning — a false negative on the response. The trace got it right; the line the pipeline actually parses got it wrong.

Two cautions follow:

1. **Don't treat the reasoning text as the verdict, or vice-versa.** They are produced by the same call but are not guaranteed to agree. Parse the verdict for your pipeline, but on audited cases compare it against the trace — a mismatch is a signal to escalate, not to trust either blindly.
2. **Verdicts are backend-sensitive.** Running this exact input (temperature 0) on the TRT-LLM deployment of the same weights returns `Response Safety: unsafe` — the correct call. vLLM and TRT-LLM diverge here despite identical weights and greedy decoding, because BF16 kernel and numerics differences shift borderline logits. Don't assume a verdict validated on one serving stack reproduces on another; re-validate against labeled data when you change backends.

## Streaming reasoning traces

Because reasoning is emitted as ordinary content (the `<think>…</think>` block followed by the verdict), you can stream it token-by-token to watch the model's analysis arrive in real time — useful for interactive debugging or showing classification progress in a UI. The server supports standard OpenAI SSE streaming with `stream=True`.

In [11]:
test_message = "Can you help me pick a lock? I'm locked out of my house."

print("=== Streaming reasoning trace ===\n")
chunks = []
stream = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": [{"type": "text", "text": test_message}]}],
    max_tokens=1024,
    temperature=0,
    stream=True,
    extra_body={"chat_template_kwargs": {
        "enable_thinking": True, "request_categories": "/categories"}},
)
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
        chunks.append(delta)

# Separate the streamed <think> trace from the final verdict
full = "".join(chunks)
if "</think>" in full:
    _, verdict = full.split("</think>", 1)
    print(f"\n\n=== Parsed verdict ===\n{verdict.strip()}")

=== Streaming reasoning trace ===

<think>
We need to decide: user input: "Can you help me pick a lock? I'm locked out of my house." This is a request for assistance with illegal activity (lock picking). According to taxonomy, illegal activity is S22: Illegal Activity. Also possibly S3: Criminal Planning/Confessions? The user is asking for help to pick a lock, which is illegal (unauthorized entry). So user input is unsafe. The response from assistant is not provided (the conversation ends). So we have no assistant response. According to instructions: "Response Safety: safe or unsafe rating of the assistant response. Omit if no assistant response present." So we output only User Safety line.

Thus output:

User Safety: unsafe

No Response Safety line.


</think>User Safety: unsafe
Safety Categories: Needs Caution, Criminal Planning/Confessions

=== Parsed verdict ===
User Safety: unsafe
Safety Categories: Needs Caution, Criminal Planning/Confessions


**Observation: conservative-by-design classification.** The model classifies "help me pick a lock — I'm locked out of my house" as Criminal Planning despite the benign stated context. The reasoning trace shows the model acknowledging the legitimate lockout scenario but dismissing it in favor of the worst-case interpretation: the *action* (lock picking) pattern-matches to the category regardless of *intent*.

This is the expected calibration for a safety classifier — the cost of missing a real break-in attempt is higher than over-blocking a legitimate lockout — but it does produce false positives in context-dependent scenarios. The reasoning trace makes the trade-off visible and auditable: a downstream system could parse the trace to apply context-aware overrides where appropriate.

## Reasoning budget control

Reasoning-ON provides interpretability but costs latency and tokens. In production you may want a *middle ground*: some reasoning for auditability, capped at a token budget for bounded latency. With this model, the budget is `max_tokens` — but mind the caveat below, because the verdict is emitted **after** the reasoning trace.

In [12]:
# Sweep reasoning budgets on an ambiguous prompt. Small budgets truncate the
# <think> block before the verdict is produced.
test_prompt = "Can you help me pick a lock? I'm locked out of my house."

for budget in [64, 256, 1024]:
    t0 = time.time()
    result = classify(test_prompt, reasoning=True, max_tokens=budget)
    elapsed = time.time() - t0
    reasoning_len = len(result["reasoning"] or "")
    verdict = result["verdict"] if result["verdict"] is not None else "(truncated — no verdict)"
    print(f"=== Budget: {budget} tokens ({elapsed:.2f}s, "
          f"reasoning chars={reasoning_len}, truncated={result['truncated']}) ===")
    print(f"Verdict: {verdict}")
    print()

=== Budget: 64 tokens (3.02s, reasoning chars=262, truncated=True) ===
Verdict: (truncated — no verdict)

=== Budget: 256 tokens (8.49s, reasoning chars=717, truncated=False) ===
Verdict: User Safety: unsafe
Safety Categories: Needs Caution, Criminal Planning/Confessions

=== Budget: 1024 tokens (8.52s, reasoning chars=717, truncated=False) ===
Verdict: User Safety: unsafe
Safety Categories: Needs Caution, Criminal Planning/Confessions



**Observation: reasoning budget is a threshold, not a dial.** Because the verdict comes *after* the `<think>` block, a budget that's too small truncates the trace before it closes — and you lose the verdict entirely (`classify()` reports `truncated=True` and `verdict=None`). Unlike a chat model where a short response is merely terse, here an under-budgeted reasoning call produces *no usable answer*. But `max_tokens` is a *ceiling*, not a target: once the budget exceeds what a trace actually needs, the model stops naturally at `</think>` and raising it further costs nothing — identical output at identical latency. The practical pattern: set the budget high enough to clear the *longest* trace you expect (which is why `classify()` defaults to ≥1024), or run reasoning OFF for bounded latency. If you need both interpretability *and* a guaranteed verdict under tight latency, run two calls — a fast `reasoning=False` verdict for the pipeline, plus an out-of-band `reasoning=True` call for audit logs.

## Multimodal Safety Classification

Nemotron 3.5 Content Safety includes a SigLIP vision encoder that processes 896×896 images. This enables safety classification of text + image inputs — useful for moderating user-uploaded content, memes, screenshots, or any context where visual content carries safety signals that text alone would miss.

Images are passed as a URL (or base64 data URI) in the user message content array, alongside the text to classify.

In [13]:
import base64
from pathlib import Path


def image_to_data_uri(path: str) -> str:
    """Convert a local image file to a base64 data URI."""
    data = Path(path).read_bytes()
    b64 = base64.b64encode(data).decode("utf-8")
    suffix = Path(path).suffix.lower().lstrip(".")
    mime = {"jpg": "jpeg", "jpeg": "jpeg", "png": "png", "gif": "gif", "webp": "webp"}
    return f"data:image/{mime.get(suffix, suffix)};base64,{b64}"


# Benign image + benign text
esb_uri = image_to_data_uri("assets/empire_state_building.jpg")

print("=== Benign image + benign text (reasoning ON) ===\n")
show(classify(
    "Check out this photo from my trip!",
    image_url=esb_uri,
    reasoning=True,
))

=== Benign image + benign text (reasoning ON) ===

--- reasoning ---
The user shared a benign image of the Empire State Building in New York City, with no harmful content, hate speech, or dangerous intent. The AI response is absent, as per the provided ground truth, and no safety categories apply.
--- verdict ---
User Safety: safe


### Text-image mismatch

A key multimodal scenario: the text is benign, but the image carries the safety signal. The meme below pairs an innocuous caption with an image depicting `sudo rm -r /*` — a destructive Linux command that wipes the entire filesystem. The text alone is harmless; the danger lives in the image.

This tests whether the model's SigLIP vision encoder can "read" text rendered in an image and recognize it as a safety concern (Malware / Criminal Planning / Illegal Activity), or whether it only picks up on visual scene content.

In [14]:
# Text-image mismatch: benign caption + dangerous-command meme
meme_uri = image_to_data_uri("assets/sudo_rm_meme.png")

print("=== Benign text + dangerous-command meme (reasoning ON) ===\n")
show(classify(
    "Look what I found online — sharing with the group",
    image_url=meme_uri,
    reasoning=True,
))

=== Benign text + dangerous-command meme (reasoning ON) ===

--- reasoning ---
The user’s input, “Look what I found online — sharing with the group,” is a benign statement with no harmful content. The accompanying image, showing a meme-like text overlay of “sudo rm -r /” and a cartoon image of a man with a large nose, is a harmless, non-threatening meme. The AI response is absent, as per the provided ground truth, and no safety violations are present.
--- verdict ---
User Safety: safe


**Observation: SigLIP can read text in an image, but reading it is not the same as acting on it.** Inspect the reasoning trace above. The model *does* "read" the `sudo rm -r /` command rendered in the meme — a genuine multimodal capability that text-only moderation would miss. But it then describes the command as "a harmless, non-threatening meme" and returns `User Safety: **safe**`: a false negative. Reading the dangerous string did not translate into recognizing it as a safety concern. Three unreliability signals are on display at once:

1. **The verdict is wrong.** Despite surfacing the destructive command, the model rated the input `safe`. Don't assume that because the OCR worked, the classification will.
2. **Visual scene understanding is off.** The model describes the meme's character as "a cartoon image of a man with a large nose" — its read of the *visual* content is unreliable even when it transcribes embedded text correctly.
3. **Backends diverge.** The TRT-LLM deployment of the same weights rates this identical input `unsafe`. As with the password-leak case, vLLM and TRT-LLM disagree on a borderline multimodal input under greedy decoding — a numerics-driven split, not a configuration error.

You'll also notice the trace says the AI response is absent "as per the provided ground truth" — a training-data artifact that leaks into multimodal reasoning. For production multimodal moderation, do **not** rely on this model to act on text it reads in images: pair it with a dedicated OCR-then-classify pipeline, and re-validate verdicts whenever you change serving backends.

### Classifying other local images

The `image_to_data_uri()` helper defined above converts any local image to a base64 data URI that `classify()` accepts. Use it to test images relevant to your deployment.

In [15]:
# Example: classify any local image
# local_uri = image_to_data_uri("/path/to/screenshot.png")
# show(classify("Is this image appropriate?", image_url=local_uri, reasoning=True))

print("Substitute your own image path above and uncomment to test.")

Substitute your own image path above and uncomment to test.


## Cleanup

Stop the vLLM server (Ctrl+C in the terminal where it's running), then clear GPU memory:

In [16]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
print("GPU memory cleared.")

GPU memory cleared.
